myanimelist analysis-: focusing on correlation and regression for pointing out  what and by how much.



cleaning process has been done in another file posted on github.- https://github.com/nvmarpita/myanimelist_ratings_analysis/blob/main/anime_cleaning.ipynb


In [2]:
import pandas as pd 
df= pd.read_csv('cleaned_data.csv', encoding = 'utf-8')

Does Episode count correlate with Score?


Hypothesis: I expect little to no correlation between Episodes and Score, because it depends more on the plot and story quality of the anime than on how many episodes it has

In [3]:
df['Episodes'].corr(df['Score'])

np.float64(0.07921177375206916)

result: This supports my hypothesis: episode count doesn't have a strong effect on how an anime is rated (0.079)

Does Duration_minutes correlate with Score?


Hypothesis: i'm not expecting any correlation between duration and score as short duration movie has also been hit mostly.

In [4]:
df['Duration_minute'].corr(df['Score'])

np.float64(0.312185566082316)

Result:his partially contradicts my hypothesis; but duration alone cant strongly predict quality (0.31)

Do certain genres have a higher average Score than others? 


Hypothesis: i think yes few genre such as 'Slice of Life', 'Suspense',could have more score than other genres as it is one of the popluar and loved genre


In [5]:
df['Score'].mean()

np.float64(6.380889625286771)

Result: my hypothesis about 'Slice of Life' 6.47 was wrong it ranks 16 out of 21, in overall average but 'Suspense' ranks third top making my half hypothesis true.



In [6]:
genre_columns = ['Action','Adventure',          
'Avant Garde',         
'Award Winning',       
'Boys Love',        
'Comedy',                
'Drama',            
'Ecchi',               
'Erotica',               
'Fantasy',              
'Girls Love',            
'Gourmet',               
'Hentai',                
'Horror',               
'Mystery',              
'Romance',               
'Sci-Fi',                
'Slice of Life',         
'Sports',                
'Supernatural',          
'Suspense' ]  

genre_avg_scores = {}                

for g in genre_columns:           
    avg = df[df[g] == 1]['Score'].mean()
    genre_avg_scores[g] = avg    


In [7]:
sorted(genre_avg_scores.items(), key=lambda item: item[1], reverse=True)

[('Award Winning', np.float64(7.296308411214953)),
 ('Mystery', np.float64(6.995093085106384)),
 ('Suspense', np.float64(6.962962962962963)),
 ('Drama', np.float64(6.850645454545455)),
 ('Romance', np.float64(6.804508733624455)),
 ('Supernatural', np.float64(6.744599686028256)),
 ('Sports', np.float64(6.722045855379189)),
 ('Action', np.float64(6.674111700783421)),
 ('Adventure', np.float64(6.673997005988024)),
 ('Gourmet', np.float64(6.62766355140187)),
 ('Girls Love', np.float64(6.591553398058253)),
 ('Fantasy', np.float64(6.591242783348526)),
 ('Sci-Fi', np.float64(6.563553651570787)),
 ('Comedy', np.float64(6.522960552268246)),
 ('Boys Love', np.float64(6.500533333333333)),
 ('Slice of Life', np.float64(6.47566722972973)),
 ('Ecchi', np.float64(6.431903520208605)),
 ('Horror', np.float64(6.148137472283812)),
 ('Erotica', np.float64(6.124418604651163)),
 ('Hentai', np.float64(6.065378839590444)),
 ('Avant Garde', np.float64(5.143931623931624))]

Does Type (TV vs Movie vs OVA, etc.) relate to Score — are movies rated differently than TV series?
hypothesis: i think yes, scoring is affected by its type, TV series are often popular than others.

In [8]:
df.groupby('Type')['Score'].mean()

Type
Movie      6.344137
Music      5.861470
ONA        6.038212
OVA        6.217557
Special    6.365714
TV         6.847144
Name: Score, dtype: float64

In [9]:
df['Members'].corr(df['Score'])

np.float64(0.3909216689875975)

In [19]:
df['Favorites'].corr(df['Score'])

np.float64(0.23545370401787646)

Does Studios matter — do certain studios' anime average higher scores?

Hypothesis: I expected Studio to have a real effect on Score, since some studios likely produce better animation, storytelling, and have stronger reputations than others.

In [20]:
df.groupby('Studios')['Score'].mean()

Studios
10Gauge                                                                              6.572500
10Gauge, Studio DURIAN                                                               7.700000
1IN                                                                                       NaN
2:10 AM Animation                                                                    6.432857
33 Collective                                                                             NaN
                                                                                       ...   
studio hb                                                                                 NaN
trenova                                                                                   NaN
ufotable                                                                             7.210580
ufotable, Shaft, A-1 Pictures, SILVER LINK., Lerche, Lay-duce, CloverWorks, Drive    7.540000
ufotable, feel., Studio Flag                        

In [21]:
df['Studios'].str.split(', ')

0               [Sunrise]
1                 [Bones]
2              [Madhouse]
3               [Sunrise]
4        [Toei Animation]
               ...       
24900                 NaN
24901                 NaN
24902                 NaN
24903                 NaN
24904                 NaN
Name: Studios, Length: 24905, dtype: object

In [22]:
df_studios = df.assign(Studios=df['Studios'].str.split(', ')).explode('Studios')

In [23]:
studio_counts = df_studios['Studios'].value_counts()
valid_studios = studio_counts[studio_counts >= 10].index
df_studios_filtered = df_studios[df_studios['Studios'].isin(valid_studios)]

In [24]:
df_studios_filtered.groupby('Studios')['Score'].mean().sort_values(ascending=False)

Studios
Animation Do                7.633000
Motion Magic                7.629091
Shuka                       7.608947
Bones                       7.323919
Kyoto Animation             7.315952
                              ...   
Onionskin                   5.250714
Blue Cat                    5.144000
Saigo no Shudan             5.120909
Liberty Animation Studio    4.820000
SAMG Entertainment               NaN
Name: Score, Length: 250, dtype: float64

Result: This supports my hypothesis, though I can't confirm why without deeper analysis into what makes those top studios perform better

Do Episodes and Duration_minute, together, predict Score and how much does each one matter once you account for the other?

In [25]:
type_dummies = pd.get_dummies(df['Type'], prefix='Type')

In [26]:
df_reg = pd.concat([df, type_dummies], axis=1)

In [27]:
df_reg[['Type','Type_TV','Type_Movie','Type_Music']].head()

,Type,Type_TV,Type_Movie,Type_Music
0,TV,True,False,False
1,Movie,False,True,False
2,TV,True,False,False
3,TV,True,False,False
4,TV,True,False,False


In [31]:
import  statsmodels. api as sm 

In [32]:

X = df_reg[['Episodes', 'Duration_minute']]
y = df_reg['Score']

combined = pd.concat([X, y], axis=1).dropna()
X = combined[['Episodes', 'Duration_minute']]
y = combined['Score']

X = sm.add_constant(X)
model = sm.OLS(y, X).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                  Score   R-squared:                       0.107
Model:                            OLS   Adj. R-squared:                  0.107
Method:                 Least Squares   F-statistic:                     938.9
Date:                Thu, 24 Sep 2026   Prob (F-statistic):               0.00
Time:                        21:26:39   Log-Likelihood:                -20113.
No. Observations:               15604   AIC:                         4.023e+04
Df Residuals:                   15601   BIC:                         4.026e+04
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                      coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------
const               6.0697      0.010    6

Result:Together, Episodes and Duration explain about 10.7% of the variation in Score. This is a low number, meaning these two factors don't explain most of why anime scores differ. This matches my earlier finding that Episodes had almost no effect and Duration had only a weak effect.

In [33]:
y.isnull().sum()

np.int64(0)

Does adding Type (TV/Movie/Music/OVA/ONA/Special) to the model improve how much of Score it explains and does Type matter once Episodes and Duration are already accounted for?

In [34]:
X = df_reg[['Episodes', 'Duration_minute', 'Type_Movie', 'Type_Music', 'Type_OVA', 'Type_ONA', 'Type_Special']]
y = df_reg['Score']

In [35]:
combined = pd.concat([X, y], axis=1).dropna()
X = combined[['Episodes', 'Duration_minute', 'Type_Movie', 'Type_Music', 'Type_OVA', 'Type_ONA', 'Type_Special']]
y = combined['Score']
X = sm.add_constant(X)
model = sm.OLS(y, X).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                  Score   R-squared:                       0.246
Model:                            OLS   Adj. R-squared:                  0.246
Method:                 Least Squares   F-statistic:                     727.0
Date:                Thu, 24 Sep 2026   Prob (F-statistic):               0.00
Time:                        21:26:39   Log-Likelihood:                -18797.
No. Observations:               15604   AIC:                         3.761e+04
Df Residuals:                   15596   BIC:                         3.767e+04
Df Model:                           7                                         
Covariance Type:            nonrobust                                         
                      coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------
const               6.5060      0.015    4

result: I expected Type to add meaningful explanatory power, since my earlier group comparison already showed TV scoring noticeably higher than other types and r went from 0.107 to 0.246 once Type was added. moderate change only.

In [36]:
X = df_reg[['Episodes', 'Duration_minute', 'Type_Movie', 'Type_Music', 'Type_OVA', 'Type_ONA', 'Type_Special', 'Mystery', 'Suspense']]
y = df_reg['Score']

combined = pd.concat([X, y], axis=1).dropna()
X = combined[['Episodes', 'Duration_minute', 'Type_Movie', 'Type_Music', 'Type_OVA', 'Type_ONA', 'Type_Special', 'Mystery', 'Suspense']]
y = combined['Score']

X = sm.add_constant(X)
model = sm.OLS(y, X).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                  Score   R-squared:                       0.253
Model:                            OLS   Adj. R-squared:                  0.252
Method:                 Least Squares   F-statistic:                     585.5
Date:                Thu, 24 Sep 2026   Prob (F-statistic):               0.00
Time:                        21:26:39   Log-Likelihood:                -18728.
No. Observations:               15604   AIC:                         3.748e+04
Df Residuals:                   15594   BIC:                         3.755e+04
Df Model:                           9                                         
Covariance Type:            nonrobust                                         
                      coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------
const               6.4883      0.015    4

In [37]:
df_reg.groupby('Type')['Members'].mean()


Type
Movie      23074.907327
Music        976.727103
ONA         7047.844325
OVA        13513.580226
Special    14427.167318
TV         92545.318810
Name: Members, dtype: float64

result: my hypothesis was correct as i expected `Mystery` and `Suspense` to remain. Mystery has P score= 0.000, Suspense has p score = 0.000 both are statistically significant (p < 0.05 is the standard cutoff). They did remain significant.
 